In [1]:
from dotenv import load_dotenv
load_dotenv("../.env")

True

In [2]:
import json
from uuid import uuid4
from pathlib import Path
from textwrap import dedent

import pandas as pd

from rich import print
from openai import OpenAI
from pydantic import BaseModel
from openai.lib._pydantic import to_strict_json_schema

In [3]:
client = OpenAI()

In [4]:
SYSTEM_PROMPTS = {
    "beir_corpus": dedent("""
                          Translate this Sundanese text including it's title and body into English.
                          Beware that it might contain a specific Sundanese context or nuances that must be correctly interpreted and not translated literally.
                          """),
    "beir_query": dedent("""
                         Translate this Sundanese with possible Indonesian text into English.
                         Beware that it might contain a specific Sundanese context or nuances that must be correctly interpreted and not translated literally.
                         """),
    "triplet": dedent("""
                      Translate this Sundanese passages into English.
                      You will be provided with a query along with the relevant and irrelevant answers.
                      Beware that it might contain a specific Sundanese context or nuances that must be correctly interpreted and not translated literally.
                      """),
}

## BEIR

### Corpus

In [5]:
df_corpus = pd.read_json("../data/cleaned/corpus.jsonl", lines=True)
df_corpus.head()

,_id,title,text
0,0f438470-de7f-47cb-8ba2-16e8b1ff5750,JEMBAR SABAR,bismillah yuga lampah balukar janglar meunang ...
1,c51ddd60-adc7-4b95-b09d-3c4865ba2aaf,KANYERI,eweuh deui seri nu lewih nyeri tibatan di héna...
2,cb4eb9c6-bf7f-4edc-b997-73a02780e60d,KARUNGING LAIN WAYAH,wanci gayuh ka peuting poék tungkeb haté lain ...
3,af9fc2c2-0230-4784-8104-cf9271ccfccc,KUDU DAÉK GAWÉ,ieu awak asa lalungsé di rasa asa carapé meure...
4,2ae6c860-37c6-47f0-aefc-8b74f96b623b,PUISI PANGGGEUING DIRI,naha anjeun téh poho yén mot téh dodoho datang...


In [6]:
def corpus_item_text(value):
    return "<title>" + value.title + "</title>\n<body>" + value.text + "</body>"

In [7]:
class BEIRCorpusItem(BaseModel):
    title_english: str
    body_english: str

In [8]:
completion_beir_corpus = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    response_format=BEIRCorpusItem,
    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPTS["beir_corpus"],
        },
        {
            "role": "user",
            "content": corpus_item_text(df_corpus.iloc[0]),
        },
    ],
)

print(completion_beir_corpus)

ParsedChatCompletion[BEIRCorpusItem](
    id='chatcmpl-BTdSRsy15HeGYg5u1Mn0n6tj4LsXH',
    choices=[
        ParsedChoice[BEIRCorpusItem](
            finish_reason='stop',
            index=0,
            logprobs=None,
            message=ParsedChatCompletionMessage[BEIRCorpusItem](
                content='{"title_english":"WIDE PATIENCE","body_english":"In the name of God, may you take action 
wisely, bearing the consequences to achieve desires. When lost, do not be disheartened; maintain a wide patience. 
Do not become stale; do not let your spirit diminish. If you are sick, do not be anxious; if you are well, do not 
be arrogant. Don\'t be little-minded. Do not be boastful, as others strive together for goodness. Do not be 
confused by the sweet temptations of the world; rather, align yourself with the greatness of God. Understand the 
true essence and examine your heart."}',
                refusal=None,
                role='assistant',
                audio=None,
                function_call=None,
                tool_calls=None,
                parsed=BEIRCorpusItem(
                    title_english='WIDE PATIENCE',
                    body_english="In the name of God, may you take action wisely, bearing the consequences to 
achieve desires. When lost, do not be disheartened; maintain a wide patience. Do not become stale; do not let your 
spirit diminish. If you are sick, do not be anxious; if you are well, do not be arrogant. Don't be little-minded. 
Do not be boastful, as others strive together for goodness. Do not be confused by the sweet temptations of the 
world; rather, align yourself with the greatness of God. Understand the true essence and examine your heart."
                ),
                annotations=[]
            )
        )
    ],
    created=1746402887,
    model='gpt-4o-mini-2024-07-18',
    object='chat.completion',
    service_tier='default',
    system_fingerprint='fp_0392822090',
    usage=CompletionUsage(
        completion_tokens=134,
        prompt_tokens=230,
        total_tokens=364,
        completion_tokens_details=CompletionTokensDetails(
            accepted_prediction_tokens=0,
            audio_tokens=0,
            reasoning_tokens=0,
            rejected_prediction_tokens=0
        ),
        prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)
    )
)

### Queries

In [9]:
df_queries = pd.read_json("../data/cleaned/queries.jsonl", lines=True)
df_queries.head()

,_id,text
0,f4ee6408-7047-498e-a319-9189cf81d378,apa maksud dari bismillah yuga lampah
1,c827cfdb-2c57-486f-abc7-80355b884699,kenapa penting sabar dalam hidup
2,0dce18d7-b0b8-4202-a1d5-683489ac423d,apa yang dimaksud dengan halangan dalam mencap...
3,5c961ec5-158a-4073-b33e-3a0be48de7a0,bagaimana cara menjadi orang yang baik menurut...
4,3a019977-ed9b-4d71-8dbd-a65c1c95cc65,apa arti dari pariksa ati


In [10]:
def beir_query_text(value):
    return "<query>" + value.text + "</query>"

In [11]:
class BEIRQueryItem(BaseModel):
    query_english: str

In [12]:
completion_beir_query = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    response_format=BEIRQueryItem,
    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPTS["beir_query"],
        },
        {
            "role": "user",
            "content": beir_query_text(df_queries.iloc[0]),
        },
    ],
)

print(completion_beir_query)

ParsedChatCompletion[BEIRQueryItem](
    id='chatcmpl-BTdSZi7X49otQmxXGyuYYBpzRBxNA',
    choices=[
        ParsedChoice[BEIRQueryItem](
            finish_reason='stop',
            index=0,
            logprobs=None,
            message=ParsedChatCompletionMessage[BEIRQueryItem](
                content='{"query_english":"What does \'Bismillah yuga lampah\' mean?"}',
                refusal=None,
                role='assistant',
                audio=None,
                function_call=None,
                tool_calls=None,
                parsed=BEIRQueryItem(query_english="What does 'Bismillah yuga lampah' mean?"),
                annotations=[]
            )
        )
    ],
    created=1746402895,
    model='gpt-4o-mini-2024-07-18',
    object='chat.completion',
    service_tier='default',
    system_fingerprint='fp_0392822090',
    usage=CompletionUsage(
        completion_tokens=20,
        prompt_tokens=107,
        total_tokens=127,
        completion_tokens_details=CompletionTokensDetails(
            accepted_prediction_tokens=0,
            audio_tokens=0,
            reasoning_tokens=0,
            rejected_prediction_tokens=0
        ),
        prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)
    )
)

## Triplet

In [13]:
df_triplet = pd.read_json("../data/cleaned/triplet.jsonl", lines=True)
df_triplet.head()

,query,positive,negative
0,Kumaha cara ngahontal kahayang dina kahirupan?,"Kahiji, urang kedah sabar sareng henteu janten...",Abdi ngadangu seueur warta ngeunaan jalma anu ...
1,Naon anu kedah dilakukeun lamun gering?,"Lamun gering, ulah rungsing sabab pikiran posi...",Masyarakat ayeuna seueur nganggur di kota nu g...
2,Kumaha cara nyieun amal?,"Ngawitan amal ti hal-hal leutik, sapertos ngab...",Jalma sering nyarita ngeunaan kaékonomian anu ...
3,Kumaha sangkan ngeterkeun diri ka Gusti?,"Mertahankeun ati, pariksa tindakan sorangan, s...","Saurang guru ngajarkeun pentingna ilmu, tapi k..."
4,Naon hartina jadi jalma leutik?,Jadi jalma leutik hartina ulah sombong sareng ...,"Dina pagelaran, anu katinggali gaduh prestasi ..."


In [14]:
def triplet_item_text(value):
    return "<query>" + value.query + "</query>\n<positive>" + value.positive + "</positive>\n<negative>" + value.negative + "</negative>"

In [15]:
class TripletItem(BaseModel):
    query_english: str
    positive_english: str
    negative_english: str

In [16]:
completion_triplet = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    response_format=TripletItem,
    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPTS["beir_query"],
        },
        {
            "role": "user",
            "content": triplet_item_text(df_triplet.iloc[0]),
        },
    ],
)

print(completion_triplet)

ParsedChatCompletion[TripletItem](
    id='chatcmpl-BTdSflGCamclsyquxx0JkZIuC3r5F',
    choices=[
        ParsedChoice[TripletItem](
            finish_reason='stop',
            index=0,
            logprobs=None,
            message=ParsedChatCompletionMessage[TripletItem](
                content='{"query_english":"How to achieve one\'s desires in life?","positive_english":"First, we 
must be patient and not lose enthusiasm in our efforts, because the consequences of striving to achieve desires 
require diligent effort.","negative_english":"I have heard a lot of news about successful people, but I never hear 
how they overcome obstacles."}',
                refusal=None,
                role='assistant',
                audio=None,
                function_call=None,
                tool_calls=None,
                parsed=TripletItem(
                    query_english="How to achieve one's desires in life?",
                    positive_english='First, we must be patient and not lose enthusiasm in our efforts, because the
consequences of striving to achieve desires require diligent effort.',
                    negative_english='I have heard a lot of news about successful people, but I never hear how they
overcome obstacles.'
                ),
                annotations=[]
            )
        )
    ],
    created=1746402901,
    model='gpt-4o-mini-2024-07-18',
    object='chat.completion',
    service_tier='default',
    system_fingerprint='fp_0392822090',
    usage=CompletionUsage(
        completion_tokens=70,
        prompt_tokens=209,
        total_tokens=279,
        completion_tokens_details=CompletionTokensDetails(
            accepted_prediction_tokens=0,
            audio_tokens=0,
            reasoning_tokens=0,
            rejected_prediction_tokens=0
        ),
        prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)
    )
)

## TODO: Generate OpenAI Batch Request

### Batch Request Generator

In [17]:
def generate_batch(df: pd.DataFrame, system_prompt: str, base_model: BaseModel, formatter_fun):
    for row in df.itertuples():
        custom_id = str(uuid4())
        job_data = {
            "custom_id": custom_id,
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": "gpt-4o-mini",
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": formatter_fun(row)},
                ],
                "response_format": {
                    "type": "json_schema",
                    "json_schema": {
                        "name": base_model.__name__,
                        "strict": True,
                        "schema": to_strict_json_schema(base_model),
                    },
                },
            },
        }
        
        yield (custom_id, str(row.Index), job_data)

In [18]:
# item = next(generate_batch(df_corpus, SYSTEM_PROMPTS["beir_corpus"], BEIRCorpusItem, corpus_item_text))
# item = next(generate_batch(df_queries, SYSTEM_PROMPTS["beir_query"], BEIRQueryItem, beir_query_text))
item = next(generate_batch(df_triplet, SYSTEM_PROMPTS["triplet"], TripletItem, triplet_item_text))

print(item)

(
    'ab739587-3f45-4dec-bde4-fd63b5915967',
    '0',
    {
        'custom_id': 'ab739587-3f45-4dec-bde4-fd63b5915967',
        'method': 'POST',
        'url': '/v1/chat/completions',
        'body': {
            'model': 'gpt-4o-mini',
            'messages': [
                {
                    'role': 'system',
                    'content': '\nTranslate this Sundanese passages into English.\nYou will be provided with a 
query along with the relevant and irrelevant answers.\nBeware that it might contain a specific Sundanese context or
nuances that must be correctly interpreted and not translated literally.\n'
                },
                {
                    'role': 'user',
                    'content': '<query>Kumaha cara ngahontal kahayang dina kahirupan?</query>\n<positive>Kahiji, 
urang kedah sabar sareng henteu janten hambar dina usaha, sabab balukar janglar pikeun ngahontal kahayang peryogi 
usaha anu tekun.</positive>\n<negative>Abdi ngadangu seueur warta ngeunaan jalma anu suksés, tapi henteu pernah 
nguping kumaha cara maranéhna ngaliwatan rintangan.</negative>'
                }
            ],
            'response_format': {
                'type': 'json_schema',
                'json_schema': {
                    'name': 'TripletItem',
                    'strict': True,
                    'schema': {
                        'properties': {
                            'query_english': {'title': 'Query English', 'type': 'string'},
                            'positive_english': {'title': 'Positive English', 'type': 'string'},
                            'negative_english': {'title': 'Negative English', 'type': 'string'}
                        },
                        'required': ['query_english', 'positive_english', 'negative_english'],
                        'title': 'TripletItem',
                        'type': 'object',
                        'additionalProperties': False
                    }
                }
            }
        }
    }
)

In [19]:
def persist_batch(df: pd.DataFrame, schema: BaseModel, format_fun, kind: str):
    batch_req_path = Path(f"../data/llm-gen/translated/{kind}_batch.jsonl")
    batch_map_path = Path(f"../data/llm-gen/translated/{kind}_map.jsonl")
    
    with open(batch_req_path, "w") as fm, open(batch_map_path, "w") as mm:
        batch_iter = generate_batch(df, SYSTEM_PROMPTS[kind], schema, format_fun)
        for custom_id, doc_id, req in batch_iter:
            json.dump(req, fm)
            fm.write("\n")

            json.dump({"custom_id": custom_id, "doc_index": doc_id}, mm)
            mm.write("\n")
    
    return batch_req_path, batch_map_path

In [20]:
beir_corpus_req_path, beir_corpus_map_path = persist_batch(df_corpus, BEIRCorpusItem, corpus_item_text, "beir_corpus")
beir_query_req_path, beir_query_map_path = persist_batch(df_queries, BEIRQueryItem, beir_query_text, "beir_query")
triplet_req_path, triplet_map_path = persist_batch(df_triplet, TripletItem, triplet_item_text, "triplet")

### Submit Batch Requests

In [21]:
def submit_batch(path):
    batch_file = client.files.create(file=open(path, "rb"), purpose="batch")

    return client.batches.create(
        input_file_id=batch_file.id, 
        endpoint="/v1/chat/completions", 
        completion_window="24h"
    )

In [22]:
triplet_batch = submit_batch(triplet_req_path.resolve())
print(triplet_batch)

Batch(
    id='batch_6817fe84fd34819084b7c63b5bc49b69',
    completion_window='24h',
    created_at=1746402948,
    endpoint='/v1/chat/completions',
    input_file_id='file-3EWAaLmfUFgXTtu1psVV58',
    object='batch',
    status='validating',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=None,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=1746489348,
    failed_at=None,
    finalizing_at=None,
    in_progress_at=None,
    metadata=None,
    output_file_id=None,
    request_counts=BatchRequestCounts(completed=0, failed=0, total=0)
)

In [23]:
beir_query_batch = submit_batch(beir_query_req_path.resolve())
print(beir_query_batch)

Batch(
    id='batch_6817fe8d33a481908971f8c1db7abc59',
    completion_window='24h',
    created_at=1746402957,
    endpoint='/v1/chat/completions',
    input_file_id='file-5uM5xHmGg3S1hHQdsoWDAw',
    object='batch',
    status='validating',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=None,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=1746489357,
    failed_at=None,
    finalizing_at=None,
    in_progress_at=None,
    metadata=None,
    output_file_id=None,
    request_counts=BatchRequestCounts(completed=0, failed=0, total=0)
)

In [24]:
beir_corpus_batch = submit_batch(beir_corpus_req_path.resolve())
print(beir_corpus_batch)

Batch(
    id='batch_68180c78133881908cb8388e303a966e',
    completion_window='24h',
    created_at=1746406520,
    endpoint='/v1/chat/completions',
    input_file_id='file-G1fyzVU2mi2NXceLGPyQbQ',
    object='batch',
    status='validating',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=None,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=1746492920,
    failed_at=None,
    finalizing_at=None,
    in_progress_at=None,
    metadata=None,
    output_file_id=None,
    request_counts=BatchRequestCounts(completed=0, failed=0, total=0)
)